# 🎬 TMDB Data Pipeline
## Step 1: Pull → Clean → Store in SQLite

> **Goal:** Fetch ~2,000 movies from TMDB and store them in a clean SQLite database with 5 tables, ready for EDA.

---

## What this notebook builds

```
movies.db
├── movies          (2,000+ rows) — title, budget, revenue, runtime, rating, year, language
├── genres          (19 rows)     — genre_id, genre_name  
├── movie_genres    (5,000+ rows) — movie_id ↔ genre_id  (many-to-many)
├── cast            (10,000+ rows)— top 5 actors per movie
└── directors       (2,000+ rows) — one director per movie
```

## Two-pass ingestion strategy

```
Pass 1: /discover/movie  (paginated, 20 movies/call)
        → get movie IDs + basic summary fields
        → fast: 100 pages = 2,000 movies in ~30 seconds

Pass 2: /movie/{id}?append_to_response=credits
        → get budget, revenue, runtime, cast, crew
        → slower: one call per movie, ~8 minutes for 2,000
        → saves progress as it goes — safe to interrupt
```

---
### ⚡ Before you start
1. Paste your TMDB API key into the `.env` file (instructions in Chapter 1)
2. Run cells **top to bottom** with `Shift + Enter`
3. The notebook saves progress after every 50 movies — safe to stop and resume

---
# 📁 Chapter 1: Setup

## Your `.env` file

Create a file called `.env` in the same folder as this notebook with this content:

```bash
TMDB_API_KEY=your_actual_key_here
```

Never put the key directly in the code. Never commit `.env` to Git.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────
!pip3 install requests python-dotenv -q
print("Done.")

Done.


In [2]:
# ── Imports ───────────────────────────────────────────────────────────
import os, json, time, sqlite3, requests
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ.get("TMDB_API_KEY")

if not API_KEY:
    raise EnvironmentError(
        "TMDB_API_KEY not found!\n"
        "Create a .env file in this folder with:\n"
        "  TMDB_API_KEY=your_key_here"
    )

print(f"API key loaded ({len(API_KEY)} characters)")
print("Ready to fetch data.")

BASE = "https://api.themoviedb.org/3"
DB   = "movies.db"


/Users/taachba5/Desktop/Personal/EDA Workshop/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


API key loaded (32 characters)
Ready to fetch data.


In [ ]:
# ── Create folder structure ───────────────────────────────────────────
Path("data/raw").mkdir(parents=True, exist_ok=True)
print("data/raw/ folder ready.")

---
# 🗄️ Chapter 2: Database Schema

We create all 5 tables upfront before fetching any data.
This is good practice — define your structure before filling it.

```sql
movies          — one row per film, all core numeric/text fields
genres          — lookup table: genre_id → genre_name  
movie_genres    — junction: many genres per movie (exploded from API list)
cast            — top 5 billed actors per movie
directors       — extracted from the crew list (job = "Director")
```

Why separate tables instead of one big CSV?
- `movie_genres` lets us `GROUP BY genre` without string splitting
- `cast` and `directors` let us ask "which actor appears in the most films?"
- Proper normalisation = no repeated data, clean JOINs


In [3]:
# ── Create SQLite database and all 5 tables ──────────────────────────
conn   = sqlite3.connect(DB)
cursor = conn.cursor()

cursor.executescript("""
    PRAGMA journal_mode=WAL;

    CREATE TABLE IF NOT EXISTS movies (
        movie_id          INTEGER PRIMARY KEY,
        title             TEXT    NOT NULL,
        original_title    TEXT,
        release_date      TEXT,
        release_year      INTEGER,
        release_month     INTEGER,
        runtime           INTEGER,
        budget            REAL,
        revenue           REAL,
        vote_average      REAL,
        vote_count        INTEGER,
        popularity        REAL,
        original_language TEXT,
        overview          TEXT,
        tagline           TEXT,
        is_franchise      INTEGER DEFAULT 0,
        collection_name   TEXT,
        fetched_at        TEXT
    );

    CREATE TABLE IF NOT EXISTS genres (
        genre_id   INTEGER PRIMARY KEY,
        genre_name TEXT NOT NULL
    );

    CREATE TABLE IF NOT EXISTS movie_genres (
        movie_id   INTEGER,
        genre_id   INTEGER,
        genre_name TEXT,
        PRIMARY KEY (movie_id, genre_id),
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
        FOREIGN KEY (genre_id) REFERENCES genres(genre_id)
    );

    CREATE TABLE IF NOT EXISTS cast (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        movie_id     INTEGER,
        actor_name   TEXT,
        character    TEXT,
        billing_order INTEGER,
        person_id    INTEGER,
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
    );

    CREATE TABLE IF NOT EXISTS directors (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        movie_id     INTEGER UNIQUE,
        director_name TEXT,
        person_id    INTEGER,
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
    );
""")

conn.commit()
print("Database created:", DB)
print()
print("Tables created:")
tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
for t in tables:
    print(f"  {t[0]}")


Database created: movies.db

Tables created:
  movies
  genres
  movie_genres
  cast
  sqlite_sequence
  directors


---
# 🏷️ Chapter 3: Fetch Genres Lookup Table

The genres table is tiny (19 rows) but important — it's the lookup table that makes the `movie_genres` JOIN meaningful.

We fetch this first because it's one call and verifies our API key works.

In [4]:
# ── Fetch and store all movie genres ─────────────────────────────────
r = requests.get(f"{BASE}/genre/movie/list",
                 params={"api_key": API_KEY},
                 timeout=10)
r.raise_for_status()

genres_data = r.json()["genres"]

# Insert into database
cursor.executemany(
    "INSERT OR IGNORE INTO genres (genre_id, genre_name) VALUES (?, ?)",
    [(g["id"], g["name"]) for g in genres_data]
)
conn.commit()

print(f"Fetched and stored {len(genres_data)} genres:\n")
for g in genres_data:
    print(f"  {g['id']:>5}  {g['name']}")


Fetched and stored 19 genres:

     28  Action
     12  Adventure
     16  Animation
     35  Comedy
     80  Crime
     99  Documentary
     18  Drama
  10751  Family
     14  Fantasy
     36  History
     27  Horror
  10402  Music
   9648  Mystery
  10749  Romance
    878  Science Fiction
  10770  TV Movie
     53  Thriller
  10752  War
     37  Western


---
# 🔍 Chapter 4: Pass 1 — Discover Movie IDs

The `/discover/movie` endpoint lets us filter movies by any criteria and returns 20 per page.

**Our strategy: fetch the most-voted movies across multiple decades**

Why `vote_count.gte=200`? Movies with very few votes have unreliable ratings. Filtering for 200+ votes gives us movies that are "real" enough to analyse.

We'll collect from 4 different sort criteria to get variety:
- Most voted (popular blockbusters)
- Highest revenue
- Highest rated (with enough votes)
- Most recent popular

This gives us ~2,000 diverse movies rather than just the top 500 most popular.

> 💡 **Rate limit:** TMDB allows ~40 requests/second. We sleep 0.25s between calls = 4 req/sec. Very safe.

In [ ]:
# ── Pass 1: Discover movie IDs across multiple criteria ───────────────

DISCOVER_CONFIGS = [
    # (sort_by, pages, label)
    ("vote_count.desc",   50, "most voted"),
    ("revenue.desc",      25, "highest revenue"),
    ("vote_average.desc", 25, "highest rated"),
    ("popularity.desc",   20, "most popular recent"),
]

all_discovered = {}  # {movie_id: basic_data} — dict deduplicates automatically

for sort_by, pages, label in DISCOVER_CONFIGS:
    print(f"Fetching '{label}' ({pages} pages)...")
    fetched = 0

    for page in range(1, pages + 1):
        try:
            r = requests.get(
                f"{BASE}/discover/movie",
                params={
                    "api_key":         API_KEY,
                    "sort_by":         sort_by,
                    "vote_count.gte":  200,
                    "page":            page,
                },
                timeout=15
            )
            r.raise_for_status()
            results = r.json().get("results", [])

            for movie in results:
                all_discovered[movie["id"]] = movie

            fetched += len(results)
            time.sleep(0.25)

        except Exception as e:
            print(f"  Error on page {page}: {e}")
            time.sleep(2)

    print(f"  Done. Running total unique movies: {len(all_discovered)}")



Fetching 'most voted' (50 pages)...
  Done. Running total unique movies: 1000
Fetching 'highest revenue' (25 pages)...
  Done. Running total unique movies: 1097
Fetching 'highest rated' (25 pages)...
  Done. Running total unique movies: 1452
Fetching 'most popular recent' (20 pages)...
  Done. Running total unique movies: 1604


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/discovered_ids.json'

In [6]:
# Save the raw discovery data
with open("data/raw/discovered_ids.json", "w") as f:
    json.dump(list(all_discovered.keys()), f)

print(f"\nTotal unique movies discovered: {len(all_discovered)}")
print(f"IDs saved to data/raw/discovered_ids.json")



Total unique movies discovered: 1604
IDs saved to data/raw/discovered_ids.json


In [7]:
# ── Preview what Pass 1 gives us ─────────────────────────────────────
# Pass 1 gives basic fields but NOT budget/revenue/runtime — those need Pass 2

sample = list(all_discovered.values())[:1]
print("Sample record from Pass 1 (what /discover returns):")
print(json.dumps(sample[0], indent=2))
print()
print("Notice: NO budget, NO revenue, NO runtime")
print("Those require a separate call to /movie/{id}")


Sample record from Pass 1 (what /discover returns):
{
  "adult": false,
  "backdrop_path": "/2ssWTSVklAEc98frZUQhgtGHx7s.jpg",
  "genre_ids": [
    12,
    18,
    878
  ],
  "id": 157336,
  "original_language": "en",
  "original_title": "Interstellar",
  "overview": "The adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space travel and conquer the vast distances involved in an interstellar voyage.",
  "popularity": 51.3713,
  "poster_path": "/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg",
  "release_date": "2014-11-05",
  "title": "Interstellar",
  "video": false,
  "vote_average": 8.468,
  "vote_count": 39133
}

Notice: NO budget, NO revenue, NO runtime
Those require a separate call to /movie/{id}


---
# 🎯 Chapter 5: Pass 2 — Full Movie Details + Credits

Now we call `/movie/{id}?append_to_response=credits` for each movie.

This gives us everything:
- `budget`, `revenue`, `runtime` (not in discover results)
- `tagline`, `overview`
- `belongs_to_collection` (franchise detection)
- Full cast list
- Full crew list (we extract the director)

**`append_to_response=credits`** is a TMDB trick — it adds the credits data to the same response, so we get movie + cast/crew in ONE API call instead of two. Very efficient.

**Progress saving:** We save to SQLite after every movie and skip movies already in the database. So if you stop and restart, it picks up where it left off.

In [8]:
# ── Helper functions for extracting and inserting data ───────────────

def extract_director(crew_list):
    """Find the director from the crew list."""
    for member in crew_list:
        if member.get("job") == "Director":
            return member.get("name"), member.get("id")
    return None, None


def parse_year_month(date_str):
    """Safely extract year and month from a date string like '1999-10-15'."""
    if not date_str:
        return None, None
    try:
        dt = datetime.strptime(date_str, "%Y-%m-%d")
        return dt.year, dt.month
    except:
        return None, None


def insert_movie(conn, data, credits):
    """Insert one movie's full data into all 5 tables."""
    cursor = conn.cursor()
    movie_id = data["id"]

    year, month = parse_year_month(data.get("release_date"))

    collection = data.get("belongs_to_collection")
    is_franchise    = 1 if collection else 0
    collection_name = collection["name"] if collection else None

    # 1. movies table
    cursor.execute("""
        INSERT OR REPLACE INTO movies (
            movie_id, title, original_title, release_date,
            release_year, release_month, runtime, budget, revenue,
            vote_average, vote_count, popularity, original_language,
            overview, tagline, is_franchise, collection_name, fetched_at
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        movie_id,
        data.get("title"),
        data.get("original_title"),
        data.get("release_date"),
        year, month,
        data.get("runtime"),
        data.get("budget"),
        data.get("revenue"),
        data.get("vote_average"),
        data.get("vote_count"),
        data.get("popularity"),
        data.get("original_language"),
        data.get("overview"),
        data.get("tagline"),
        is_franchise,
        collection_name,
        datetime.now().isoformat()
    ))

    # 2. movie_genres junction table
    for genre in data.get("genres", []):
        cursor.execute(
            "INSERT OR IGNORE INTO movie_genres (movie_id, genre_id, genre_name) VALUES (?,?,?)",
            (movie_id, genre["id"], genre["name"])
        )

    # 3. cast table — top 5 billed actors only
    for actor in credits.get("cast", [])[:5]:
        cursor.execute(
            "INSERT OR IGNORE INTO cast (movie_id, actor_name, character, billing_order, person_id) VALUES (?,?,?,?,?)",
            (movie_id, actor.get("name"), actor.get("character"),
             actor.get("order"), actor.get("id"))
        )

    # 4. directors table
    director_name, director_id = extract_director(credits.get("crew", []))
    if director_name:
        cursor.execute(
            "INSERT OR REPLACE INTO directors (movie_id, director_name, person_id) VALUES (?,?,?)",
            (movie_id, director_name, director_id)
        )


def already_fetched(conn, movie_id):
    """Check if this movie is already in the database."""
    result = conn.execute(
        "SELECT 1 FROM movies WHERE movie_id=?", (movie_id,)
    ).fetchone()
    return result is not None


print("Helper functions ready.")


Helper functions ready.


In [9]:
# ── Pass 2: Fetch full details for every discovered movie ─────────────
# This is the slow step — one API call per movie.
# At 2,000 movies with 0.25s sleep = ~8 minutes.
# Progress is saved to SQLite every movie — safe to stop and restart.

movie_ids  = list(all_discovered.keys())
total      = len(movie_ids)
success    = 0
skipped    = 0
errors     = 0

print(f"Fetching details for {total} movies...")
print(f"Estimated time: ~{total * 0.25 / 60:.0f} minutes")
print()

for i, movie_id in enumerate(movie_ids):

    # Skip movies already in the database (safe restart behaviour)
    if already_fetched(conn, movie_id):
        skipped += 1
        continue

    try:
        r = requests.get(
            f"{BASE}/movie/{movie_id}",
            params={
                "api_key":             API_KEY,
                "append_to_response":  "credits",
            },
            timeout=15
        )

        # Skip 404s — movie was removed from TMDB
        if r.status_code == 404:
            errors += 1
            continue

        r.raise_for_status()
        data    = r.json()
        credits = data.pop("credits", {"cast": [], "crew": []})

        insert_movie(conn, data, credits)
        conn.commit()
        success += 1

        # Progress update every 100 movies
        if success % 100 == 0:
            pct = (i + 1) / total * 100
            print(f"  [{pct:>5.1f}%]  {success} fetched  {skipped} skipped  {errors} errors")

        time.sleep(0.25)

    except requests.exceptions.Timeout:
        print(f"  Timeout on movie {movie_id} — skipping")
        errors += 1
        time.sleep(1)

    except requests.exceptions.HTTPError as e:
        if "429" in str(e):
            print("  Rate limit hit — sleeping 10s")
            time.sleep(10)
        else:
            errors += 1

    except Exception as e:
        errors += 1
        time.sleep(0.5)

print()
print(f"Pass 2 complete!")
print(f"  Fetched:  {success:,}")
print(f"  Skipped:  {skipped:,} (already in db)")
print(f"  Errors:   {errors:,}")


Fetching details for 1604 movies...
Estimated time: ~7 minutes

  [  6.2%]  100 fetched  0 skipped  0 errors
  [ 12.5%]  200 fetched  0 skipped  0 errors
  [ 18.7%]  300 fetched  0 skipped  0 errors
  [ 24.9%]  400 fetched  0 skipped  0 errors
  [ 31.2%]  500 fetched  0 skipped  0 errors
  [ 37.4%]  600 fetched  0 skipped  0 errors
  [ 43.6%]  700 fetched  0 skipped  0 errors
  [ 49.9%]  800 fetched  0 skipped  0 errors
  [ 56.1%]  900 fetched  0 skipped  0 errors
  [ 62.3%]  1000 fetched  0 skipped  0 errors
  [ 68.6%]  1100 fetched  0 skipped  0 errors
  [ 74.8%]  1200 fetched  0 skipped  0 errors
  [ 81.0%]  1300 fetched  0 skipped  0 errors
  [ 87.3%]  1400 fetched  0 skipped  0 errors
  [ 93.5%]  1500 fetched  0 skipped  0 errors
  [ 99.8%]  1600 fetched  0 skipped  0 errors

Pass 2 complete!
  Fetched:  1,604
  Skipped:  0 (already in db)
  Errors:   0


---
# ✅ Chapter 6: Verify the Database

Before doing any EDA, always verify the database looks right. Check row counts, sample data, and basic statistics to confirm the ingestion worked.

In [10]:
# ── Row counts for every table ────────────────────────────────────────
print("Database:", DB)
print()

tables = ["movies", "genres", "movie_genres", "cast", "directors"]
for table in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<15} {count:>7,} rows")


Database: movies.db

  movies            1,604 rows
  genres               19 rows
  movie_genres      4,456 rows
  cast              7,937 rows
  directors         1,602 rows


---
# 📋 Chapter 7: What You Now Have

## Database summary

```
movies.db
├── movies          ~2,000 rows  — title, year, budget, revenue, runtime, rating, language
├── genres              19 rows  — genre_id, genre_name  
├── movie_genres    ~5,000 rows  — movie_id ↔ genre_id
├── cast           ~10,000 rows  — top 5 actors per movie
└── directors       ~2,000 rows  — director per movie
```

## What this unlocks for EDA

| Question | Tables needed |
|----------|--------------|
| Which genres earn the most? | `movies` JOIN `movie_genres` |
| Does bigger budget = more revenue? | `movies` only |
| Which directors have best ROI? | `movies` JOIN `directors` |
| What's the most common actor-genre combo? | `cast` JOIN `movie_genres` |
| How has runtime changed over decades? | `movies` only |
| Franchise vs standalone performance? | `movies` (is_franchise column) |
| Best month to release a film? | `movies` (release_month column) |

## Known data quality issues (for the cleaning chapter)

- `budget = 0` → means **unreported**, not literally zero → replace with `NaN`
- `revenue = 0` → same issue
- Some movies have no director (documentaries, multi-director films)
- Runtime can be `NULL` for some entries
- Very old films (pre-1970) have sparse data


